# Cionic Metrics Analysis

This notebook performs gait metrics extraction and visualization from several data gait collections. The analysis includes:

- **Metrics Extraction**: Compute statistical measures (mean, std, CV) from sensor streams
- **Data Visualization**: Generate violin plots and statistical summary charts
- **Group Comparison**: Compare metrics across different experimental groups
- **Export Results**: Save metrics data to CSV for further analysis

### Input Requirements
- **Metadata List**: Defines experimental groups and their associated recordings
- **Stream Definitions**: Specifies which sensor streams and components to analyze
- **Authentication Token**: Provides access to the Cionic API

In [ ]:
import copy
import os
from pprint import pprint

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

from cionic.gait_metrics import (
    STREAM_DEFINITIONS_FOR_STANDARD_METRICS,
    MetricsExtractor,
)
from cionic.plotting import (
    PLOTTING_METRIC_SPECIFICATION_LIST,
    TOP_RELEVANT_PLOTTING_METRICS,
    GroupedMetricsPlotter,
    StreamsPlotter,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

%load_ext autoreload
%autoreload 2

## Prepare for plots outputs
User should select desired `output_plots_format` and `dirpath`.

In [ ]:
# Options for output_plots_format:
# - "png": plots are individually saved as png
# - "pdf": all plots of a given type are saved to a single pdf
# - None: no plots are saved
output_plots_format = "png"

dirpath = "plots"  # Folder location for saved plots

assert output_plots_format in ["png", "pdf", None], "output_plots_format is invalid"
if output_plots_format:
    os.makedirs(dirpath, exist_ok=True)

## Configuration Parameters

The following parameters define the analysis configuration. When executed via papermill (automated execution), these values are automatically overwritten with user-specified parameters from the metadata creation interface.

The `metadata_list` defines the experimental groups for comparison. Each group contains:
- **group_name**: Display name for the experimental condition
- **org_shortname**: Organization identifier (typically "cionic")
- **group_color**: Color for visualization (e.g., "steelblue", "sandybrown")
- **recordings**: List of specific data collections to include

Each recording specifies:
- **study_shortname**: Name of the study protocol
- **collection_num**: Unique identifier for the data collection session
- **label**: Specific activity or condition within the collection

In [ ]:
#######################################################################################
#
# Input values:
#     - metadata_list (overwritten by papermill)
#     - figure_title (overwritten by papermill)
#     - tokenpath (overwritten by papermill)
#     - stream_definitions
#
#######################################################################################

metadata_list = [
    {  # Can create an arbitrary number of groups
        "group_name": "Unstimulated",
        "org_shortname": "cionic",
        "group_color": "steelblue",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 55,
                "label": "unstimulated_walk",
                "sides": ["left", "right"],
            },
        ],
    },
    {
        "group_name": "Stimulated",
        "org_shortname": "cionic",
        "group_color": "sandybrown",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 59,
                "label": "stim_walk",
                "sides": ["left", "right"],
            },
        ],
    },
]

tokenpath = "/home/jovyan/cionic-data/token.json"
figure_title = "Example"

## Stream Data Visualization

Plot sensor streams for each recording to visualize the underlying gait data. This section generates time-series plots showing **shank Euler angles** (sagittal and transverse planes) 

Plots are saved based on the `output_plots_format` setting and provide insight into data quality and gait patterns before metrics extraction.

In [ ]:
if output_plots_format == "pdf":
    pdf_ctx = PdfPages(f"{dirpath}/streams_plots.pdf")

for metadata in metadata_list:
    for recording in metadata["recordings"]:
        plotter = StreamsPlotter(
            org_shortname=metadata["org_shortname"],
            study_shortname=recording["study_shortname"],
            collection_num=recording["collection_num"],
            tokenpath=tokenpath,
        )

        for side in recording["sides"]:
            fig, axs = plotter.subplots(nrows=2, ncols=1)
            title = (
                f"{metadata['group_name']} | {recording['study_shortname']} "
                f"{recording['collection_num']}"
            )
            plotter.plot_stream(
                ax=axs[0],
                label=recording["label"],
                label_name=f"{side} shank sagittal",
                position=f"{side[0]}_shank",
                stream_name="euler",
                component="x",
                color=metadata["group_color"],
                title=title,
                x_label=None,
            )
            plotter.plot_stream(
                ax=axs[1],
                label=recording["label"],
                label_name=f"{side} shank transverse",
                position=f"{side[0]}_shank",
                stream_name="euler",
                component="z",
                color=metadata["group_color"],
                title=None,
            )
            for ax in axs:
                kwargs = {
                    "ax": ax,
                    "label": recording["label"],
                    "position": f"{side[0]}_shank",
                }
                plotter.plot_stride_splits(**kwargs)
                plotter.shade_walking_periods(**kwargs)
                plotter.clip_non_gait_edges(**kwargs)

            # Additional plotting customization can be added here.
            plt.tight_layout()

            if output_plots_format == "png":
                leaf = (
                    f"{recording['study_shortname']}_{recording['collection_num']}_"
                    f"{recording['label']}_{side}"
                )
                fig.savefig(
                    f"{dirpath}/streams_plot_{leaf}.png", dpi=300, bbox_inches="tight"
                )
                print(f"Saved {os.path.abspath(dirpath)}/streams_plot_{leaf}.png")
                plt.show()
            elif output_plots_format == "pdf":
                pdf_ctx.savefig(fig)
                plt.show()
                plt.close(fig)
            else:
                plt.show()

if output_plots_format == "pdf":
    pdf_ctx.close()
    print(f"Saved {os.path.abspath(dirpath)}/streams_plots.pdf")

## Stream Definitions Configuration

Stream definitions specify which sensor data streams to analyze for metrics extraction. Each definition includes:
- **stream**: Type of sensor data (e.g., 'euler', 'gyro', 'accel')
- **position**: Anatomical location (e.g., 'thigh', 'shank', 'knee_joint')
- **component**: Specific axis or measurement (e.g., 'x', 'y', 'z', 'knee_flexion')

Additional stream definitions can be added by appending dictionaries of the form:
```python
{"stream": "euler", "position": "thigh", "component": "y"}
```

In [ ]:
stream_definitions = STREAM_DEFINITIONS_FOR_STANDARD_METRICS

# Example: uncomment below to append an additional (nonstandard) stream definition
# definition = {"stream": "euler", "position": "thigh", "component": "y"}
# stream_definitions.append(definition)

pprint(stream_definitions)

## Metrics Extraction

Create the `MetricsExtractor` instance and compute gait metrics from the configured data streams. This process:

1. **Loads Data**: Retrieves sensor data for each specified recording
2. **Processes Streams**: Applies signal processing to extract meaningful metrics
3. **Aggregates Results**: Combines metrics across all groups and recordings into a single DataFrame

In [ ]:
metric_extractor = MetricsExtractor(
    metadata_list=metadata_list,
    stream_definitions=stream_definitions,
    tokenpath=tokenpath,
)
all_metrics = metric_extractor.extract_metrics()
print(f"Total rows: {len(all_metrics)}")
print(f"Total columns: {len(all_metrics.columns)}")

display(all_metrics.head())

## Data Export

Save the extracted metrics to a CSV file for further analysis and provide a summary of the results:

In [ ]:
# Create filename with timestamp
csv_filename = "metrics.csv"

# Save to CSV
all_metrics.to_csv(csv_filename, index=False)
print(f"Metrics saved to: {csv_filename}")

## Visualization Configuration

Configure which metrics to visualize using predefined plotting specifications. Each metric specification includes:

- **title**: Descriptive figure title for display
- **y_label**: Units and measurement description for the y-axis
- **metric_column**: Column name from the metrics DataFrame (e.g., 'mean_value', 'std_value', 'cv_value')
- **position**: Anatomical position being analyzed
- **component**: Specific measurement component or axis

New metrics can be added to the visualization list by appending a dictionary of the form:
```python
new_metric = {
    "title": "Thigh Mean (Sagittal)",  # Descriptive figure title
    "y_label": "Euler, (degrees)",  # Figure y label
    "metric_column": "mean_value",  # Valid metric column in all_metrics
    "position": "thigh",  # Valid position, e.g., thigh, knee_joint, etc.
    "component": "x",  # Valid component, e.g., x, y, knee_flexion
}
```

In [ ]:
metric_specification_list = copy.deepcopy(PLOTTING_METRIC_SPECIFICATION_LIST)

# Comment this out to plot broader metrics list
metric_specification_list = [
    m for m in metric_specification_list if m["title"] in TOP_RELEVANT_PLOTTING_METRICS
]

# Example: uncomment below to append an additional metric for plotting
# new_metric = {
#     "title": "Thigh Mean (Sagittal)",  # Descriptive figure title
#     "y_label": "Euler, (degrees)",  # Figure y label
#     "metric_column": "mean_value",  # Valid metric column in all_metrics
#     "position": "thigh",  # Valid position, e.g., thigh, knee_joint, etc.
#     "component": "x",  # Valid component, e.g., x, y, knee_flexion
# }
# metric_specification_list.append(new_metric)

print("Plotting metrics: ")
for spec in metric_specification_list:
    filename = spec["title"].lower().replace(" ", "_").replace("(", "").replace(")", "")
    if figure_title is not None:
        spec["title"] = f"{figure_title}  |  {spec['title']}"
        title_clean = figure_title.lower().replace(' ', '_')
        title_clean = title_clean.replace('(', '').replace(')', '')
        filename = f"{title_clean}_{filename}"
    spec["filename"] = filename
    print(f"- {spec['title']}")

## Violin Plot Visualization

Generate violin plots to show the distribution of metrics across experimental groups. Violin plots display:

- **Distribution Shape**: Full probability density of the data
- **Quartiles**: Box plot elements showing median and quartile boundaries  
- **Group Comparisons**: Side-by-side comparison of different experimental conditions
- **Individual Data Points**: Optional overlay of actual measurements

These plots are ideal for comparing variability and central tendencies between groups.

In [ ]:
if output_plots_format == "pdf":
    pdf_ctx = PdfPages(f"{dirpath}/violin_plots.pdf")

plotter = GroupedMetricsPlotter(metrics=all_metrics)
for spec in metric_specification_list:
    fig, ax = plotter.violin_plot(metric_specification=spec)
    if fig is None or ax is None:
        continue
    # Additional plotting customization can be added here.
    ax.xaxis.set_tick_params(rotation=0)
    plt.tight_layout()

    if output_plots_format == "png":
        filepath = f"{dirpath}/violin_plot_{spec['filename']}.png"
        fig.savefig(filepath, dpi=300, bbox_inches="tight")
        print(f"Saved png: {os.path.abspath(filepath)}")
        plt.show()
    elif output_plots_format == "pdf":
        pdf_ctx.savefig(fig)
        plt.show()
        plt.close(fig)
    else:
        plt.show()

if output_plots_format == "pdf":
    pdf_ctx.close()
    print(f"Saved pdf: {os.path.abspath(dirpath)}/violin_plots.pdf")

## Statistical Summary Bar Charts

Create bar charts showing statistical summaries for each metric and group. Available statistics include:

- **"mean"**: Average value across all measurements in each group
- **"std"**: Standard deviation showing absolute variability
- **"cv"**: Coefficient of variation (std/mean) showing relative variability as a percentage

These charts provide a quantitative comparison of central tendencies and variability between experimental groups.

In [ ]:
# Options: "mean", "std", "cv"
statistic = "cv"

if output_plots_format == "pdf":
    pdf_ctx = PdfPages(f"{dirpath}/bar_plots.pdf")

for spec in metric_specification_list:
    fig, ax = plotter.statistical_summary_bar_plot(
        metric_specification=spec, statistic=statistic
    )
    if fig is None or ax is None:
        continue
    # Additional plotting customization can be added here.
    ax.xaxis.set_tick_params(rotation=0)
    plt.tight_layout()

    if output_plots_format == "png":
        filepath = f"{dirpath}/bar_plot_{spec['filename']}.png"
        fig.savefig(filepath, dpi=300, bbox_inches="tight")
        print(f"Saved png: {os.path.abspath(filepath)}")
        plt.show()
    elif output_plots_format == "pdf":
        pdf_ctx.savefig(fig)
        plt.show()
        plt.close(fig)
    else:
        plt.show()

if output_plots_format == "pdf":
    pdf_ctx.close()
    print(f"Saved pdf: {os.path.abspath(dirpath)}/bar_plots.pdf")